# 05 - Black-Litterman

Start from market-implied equilibrium returns, layer in a couple of subjective views, and look at the posterior weights.

In [ ]:
import numpy as np

from markowitz import BlackLitterman, MeanVariance

rng = np.random.default_rng(11)
n = 6

# Market-cap weights (must sum to 1).
w_mkt = np.array([0.30, 0.20, 0.15, 0.15, 0.12, 0.08])

# Plausible covariance: factor + idiosyncratic.
loadings = rng.uniform(0.4, 1.2, size=n)
Sigma = np.outer(loadings, loadings) * 0.04 + np.diag(rng.uniform(0.02, 0.05, n))
delta = 2.5  # risk-aversion implied by the market
tau = 0.05

## Reverse-engineer the equilibrium returns

In [ ]:
pi = delta * Sigma @ w_mkt
print("equilibrium implied returns:")
print(np.round(pi, 4))

## Express two views

**View 1.** Asset 0 outperforms asset 1 by 2%.

**View 2.** Asset 4 returns 8%.

In [ ]:
P = np.array(
    [
        [1.0, -1.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
    ]
)
q = np.array([0.02, 0.08])

## Compute the posterior

In [ ]:
bl = BlackLitterman(
    prior_returns=pi,
    sigma=Sigma,
    P=P,
    q=q,
    tau=tau,
    omega="he-litterman",
).fit()

print("posterior mean:")
print(np.round(bl.posterior_mean_, 4))

## Compare market weights to BL-optimal weights

In [ ]:
w_bl = (
    MeanVariance(risk_aversion=delta, long_only=False)
    .fit(bl.posterior_mean_, bl.posterior_cov_)
    .weights_
)

for i in range(n):
    print(f"asset {i}: market={w_mkt[i]:+.3f}  BL={w_bl[i]:+.3f}  tilt={w_bl[i] - w_mkt[i]:+.3f}")

## Reading the tilts

The portfolio tilts away from the market in exactly the directions spanned by the rows of $P$, scaled by the view strengths and the prior confidence $\tau$. The next notebook benchmarks all of these methods out-of-sample.